In [1]:
# =========================
# 1) Imports
# =========================
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import numpy as np

# (Opcional, para heatmap bonito)
try:
    import seaborn as sns
    HAS_SNS = True
except:
    HAS_SNS = False


# =========================
# 2) Lista de animales (puedes editar)
# =========================
animals = [
    "dog", "cat", "gorilla",
    "lion", "tiger", "wolf", "fox",
    "horse", "cow", "pig",
    "elephant", "giraffe", "zebra",
    "monkey", "chimpanzee",
    "bear", "rabbit", "mouse",
    "eagle", "shark", "dolphin"
]

# Si quieres probar plural/sinónimos típicos:
# animals += ["dogs", "cats", "canine", "feline"]


# =========================
# 3) Helpers
# =========================
def cos(a, b):
    return cosine_similarity([a], [b])[0, 0]

def get_vectors(words, model):
    """Devuelve (valid_words, vectors) filtrando los que no existen en el modelo."""
    valid_words = []
    vectors = []
    for w in words:
        if w in model:
            valid_words.append(w)
            vectors.append(model[w])
        else:
            print(f"⚠️ Omitido (no está en el vocab): {w}")
    return valid_words, np.array(vectors)

def cosine_matrix(vectors):
    return cosine_similarity(vectors)

def print_top_similarities(words, vectors, target, topn=8):
    if target not in words:
        print(f"'{target}' no está disponible (no está en vocab o no está en la lista).")
        return
    idx = words.index(target)
    sims = cosine_similarity([vectors[idx]], vectors)[0]
    pairs = [(words[i], float(sims[i])) for i in range(len(words)) if i != idx]
    pairs.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTop {topn} más similares a '{target}':")
    for w, s in pairs[:topn]:
        print(f"  {w:12s}  sim={s:.3f}")


# =========================
# 4) Cargar vectores (model debe existir)
# =========================
# Asumo que YA tienes cargado tu modelo, como en tu ejemplo:
# model = KeyedVectors.load_word2vec_format(...)

valid_animals, vectors = get_vectors(animals, model)

print(f"\n✅ Animales válidos: {len(valid_animals)}/{len(animals)}")
print(valid_animals)


# =========================
# 5) Similitudes: heatmap + top similares
# =========================
sim_mat = cosine_matrix(vectors)

# Heatmap
plt.figure(figsize=(12, 10))
if HAS_SNS:
    sns.heatmap(sim_mat, xticklabels=valid_animals, yticklabels=valid_animals, cmap="viridis")
else:
    plt.imshow(sim_mat, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(valid_animals)), valid_animals, rotation=90)
    plt.yticks(range(len(valid_animals)), valid_animals)
plt.title("Cosine Similarity Heatmap — Animals")
plt.tight_layout()
plt.show()

# Top similares de algunos targets
for target in ["dog", "cat", "gorilla"]:
    print_top_similarities(valid_animals, vectors, target, topn=10)


# =========================
# 6) PCA 3D (como tu ejemplo)
# =========================
pca = PCA(n_components=3)
reduced = pca.fit_transform(vectors)

coords = {word: reduced[i] for i, word in enumerate(valid_animals)}

fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection="3d")

# puntos
for word, vec in coords.items():
    ax.scatter(vec[0], vec[1], vec[2])
    ax.text(vec[0], vec[1], vec[2], word, fontsize=9)

# Conecta dog-cat-gorilla si existen
def connect(a, b, color="red"):
    if a in coords and b in coords:
        va, vb = coords[a], coords[b]
        ax.plot([va[0], vb[0]], [va[1], vb[1]], [va[2], vb[2]], color=color)
        mid = (va + vb) / 2.0
        # similitud real en el espacio original (no PCA)
        ax.text(mid[0], mid[1], mid[2],
                f"sim={cos(model[a], model[b]):.2f}", color=color)

connect("dog", "cat", color="red")
connect("dog", "gorilla", color="blue")
connect("cat", "gorilla", color="green")

ax.set_title("3D PCA — Animal Word Embeddings")
ax.set_xlabel("PCA 1"); ax.set_ylabel("PCA 2"); ax.set_zlabel("PCA 3")
plt.show()


# =========================
# 7) (Bonus) Analogías tipo "king-man+woman"
# =========================
# Ejemplo: dog - puppy + kitten ≈ cat (si existen)
def analogy(a, b, c, topn=5):
    """
    Devuelve palabras similares a: a - b + c
    """
    for w in [a,b,c]:
        if w not in model:
            print(f"⚠️ No existe en vocab: {w}")
            return
    v = model[a] - model[b] + model[c]
    # topn+1 por si sale el mismo término
    sims = model.similar_by_vector(v, topn=topn+3)
    # filtra a,b,c si aparecen
    filtered = [(w,s) for (w,s) in sims if w not in [a,b,c]]
    print(f"\nAnalogía: {a} - {b} + {c} ≈ ?")
    for w,s in filtered[:topn]:
        print(f"  {w:12s}  sim={s:.3f}")

analogy("dog", "puppy", "kitten", topn=8)
analogy("cat", "kitten", "puppy", topn=8)

NameError: name 'model' is not defined